In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report
import torch
from torch.utils.data import DataLoader
import os
import pickle
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from src.algorithms import *

In [3]:
data_full = pd.read_csv("data_w_errors.csv")

In [4]:
data_fullw = data_full[["alpha", "delta", "Vx_set", "K_set", "Vx", "Vy", "wz", "K", "s_Vx", "s_wz", "s_K"]]

In [5]:
data_fullw.shape

(8640000, 11)

In [24]:
X_train, X_test, Y_train, Y_test = (
    train_test_split(data_fullw.drop(columns=['s_Vx','s_wz', 's_K']),
                     data_fullw[['s_Vx', 's_wz', 's_K']], shuffle=True)
)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
display(device)

device(type='cuda')

In [73]:
# Преобразование в тензоры PyTorch
X_train_tensor = torch.FloatTensor(X_train.values).to(device)
y_train_tensor = torch.FloatTensor(Y_train.values).to(device)
X_test_tensor = torch.FloatTensor(X_test.values).to(device)
y_test_tensor = torch.FloatTensor(Y_test.values).to(device)

In [74]:
# 3. Инициализация модели, определение функции потерь и оптимизатора
input_size = X_train.shape[1]
model = MultiTargetSimpleNN(input_size).to(device)
criterion = nn.BCELoss()  # Бинарная кросс-энтропия для мультитаргетной классификации
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [75]:
# 4. Обучение модели
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/1000], Loss: 0.7828
Epoch [20/1000], Loss: 0.5576
Epoch [30/1000], Loss: 0.5294
Epoch [40/1000], Loss: 0.5115
Epoch [50/1000], Loss: 0.4981
Epoch [60/1000], Loss: 0.4911
Epoch [70/1000], Loss: 0.4833
Epoch [80/1000], Loss: 0.4768
Epoch [90/1000], Loss: 0.4715
Epoch [100/1000], Loss: 0.4662
Epoch [110/1000], Loss: 0.4607
Epoch [120/1000], Loss: 0.4544
Epoch [130/1000], Loss: 0.4482
Epoch [140/1000], Loss: 0.4427
Epoch [150/1000], Loss: 0.4369
Epoch [160/1000], Loss: 0.4308
Epoch [170/1000], Loss: 0.4244
Epoch [180/1000], Loss: 0.4186
Epoch [190/1000], Loss: 0.4130
Epoch [200/1000], Loss: 0.4073
Epoch [210/1000], Loss: 0.4021
Epoch [220/1000], Loss: 0.3968
Epoch [230/1000], Loss: 0.3917
Epoch [240/1000], Loss: 0.3862
Epoch [250/1000], Loss: 0.3810
Epoch [260/1000], Loss: 0.3751
Epoch [270/1000], Loss: 0.3699
Epoch [280/1000], Loss: 0.3649
Epoch [290/1000], Loss: 0.3599
Epoch [300/1000], Loss: 0.3579
Epoch [310/1000], Loss: 0.3531
Epoch [320/1000], Loss: 0.3478
Epoch [330/1000],

In [76]:
# 5. Оценка модели
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    predicted = (test_outputs > 0.5).float()  # Применяем порог 0.5 для бинарной классификации
    
    # Вычисление точности для каждого класса
    accuracy_per_class = (predicted == y_test_tensor).float().mean(dim=0)
    print(f'Accuracy per class: {accuracy_per_class.cpu().numpy()}')

Accuracy per class: [0.92359906 0.8870486  0.948831  ]


In [77]:
print(classification_report(Y_test, predicted.cpu().numpy()))

              precision    recall  f1-score   support

           0       0.89      0.71      0.79    432115
           1       0.87      0.56      0.68    464495
           2       0.93      0.79      0.86    420039

   micro avg       0.90      0.68      0.78   1316649
   macro avg       0.90      0.69      0.78   1316649
weighted avg       0.89      0.68      0.77   1316649
 samples avg       0.42      0.42      0.42   1316649



In [78]:
torch.save(model.state_dict(), "Simple_model.pth")

In [40]:
class MultiTargetRNN(nn.Module):
    def __init__(self, input_size):
        super(MultiTargetRNN, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(in_features=64, out_features=3)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out)
        return self.sigmoid(out)
        

In [61]:
# Изменение формы входных данных для RNN
X_train_tensor = X_train_tensor.view(-1, 1, X_train.shape[1]).contiguous()  # (batch_size, seq_length, input_size)
X_test_tensorRNN = X_test_tensor.view(-1, 1, X_test.shape[1]).contiguous()      # (batch_size, seq_length, input_size)

In [42]:
# 3. Инициализация модели, определение функции потерь и оптимизатора
input_size = X_train.shape[1]
modelRNN = MultiTargetRNN(input_size).to(device)
criterion = nn.BCELoss()  # Бинарная кросс-энтропия для мультитаргетной классификации
optimizer = optim.Adam(modelRNN.parameters(), lr=0.001)

In [44]:
# Установите размер батча
batch_size = 32
num_epochs = 100
accumulation_steps = 4

In [45]:
# Обучение модели
for epoch in range(num_epochs):
    modelRNN.train()
    optimizer.zero_grad()
    
    for i in range(0, len(X_train_tensor), batch_size):
        inputs = X_train_tensor[i:i+batch_size]
        labels = y_train_tensor[i:i+batch_size]
        
        outputs = modelRNN(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        if (i // batch_size + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
    
    # Освобождение памяти
    torch.cuda.empty_cache()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.0678
Epoch [20/100], Loss: 0.0657
Epoch [30/100], Loss: 0.0646
Epoch [40/100], Loss: 0.0805
Epoch [50/100], Loss: 0.0750
Epoch [60/100], Loss: 0.0651
Epoch [70/100], Loss: 0.0744
Epoch [80/100], Loss: 0.0643
Epoch [90/100], Loss: 0.0628
Epoch [100/100], Loss: 0.0639


In [62]:
print("Оригинальный размер:", X_test_tensor.shape)
print("Размер для RNN:", X_test_tensorRNN.shape)

Оригинальный размер: torch.Size([2160000, 8])
Размер для RNN: torch.Size([2160000, 1, 8])


In [65]:
# Ensure tensor is contiguous and has the correct type
if not X_test_tensorRNN.is_contiguous():
    X_test_tensorRNN = X_test_tensorRNN.contiguous()

X_test_tensorRNN = X_test_tensorRNN.float()  # Adjust as necessary

# Pass to model
try:
    modelRNN.eval()
    with torch.no_grad():
        test_outputs = modelRNN(X_test_tensorRNN[0:32])
        predicted = (test_outputs > 0.5).float()  # Применяем порог 0.5 для бинарной классификации
        
        # Вычисление точности для каждого класса
        accuracy_per_class = (predicted == y_test_tensor).float().mean(dim=0)
        print(f'Accuracy per class: {accuracy_per_class.cpu().numpy()}')
except RuntimeError as e:
    print(f"Error encountered: {e}")

Accuracy per class: [[0.8125  0.28125 0.75   ]
 [0.8125  0.28125 0.75   ]
 [0.8125  0.71875 0.75   ]
 ...
 [0.1875  0.71875 0.75   ]
 [0.8125  0.71875 0.75   ]
 [0.8125  0.28125 0.75   ]]


In [72]:
torch.save(modelRNN.state_dict(), "modelRNN.pth")

In [ ]:
print(classification_report(Y_test[0:32], predicted.cpu().numpy()))